# 7. tēma — datu analīze un vizualizācija

[![Atvērt Google Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ValRCS/RTU_BDAA_Course_2026/blob/main/notebooks/lecture_07_data_analysis_visualization/07_data_analysis_visualization.ipynb)

Turpinām 6. lekcijā sākto SS.com dzīvokļu sludinājumu projektu:

**SS.com → scraping → CSV → Pandas → tīrīšana → analīze → Matplotlib → secinājumi → tīrs CSV**

Mērķi: pārbaudīt datu kvalitāti, konvertēt tipus, filtrēt un kārtot, lietot `groupby()`/`agg()`, veidot histogrammas, stabiņu diagrammas, scatter plot un boxplot, atrast neparastus ierakstus un izveidot atkārtojamu tīrīšanas procesu.

## 0. Darba vide

Notebook paredzēts **Run All** gan VS Code, gan Google Colab.

**Lokāli:** klonējiet repozitoriju, izveidojiet virtuālo vidi un instalējiet:
```bash
python -m pip install jupyter ipykernel pandas matplotlib plotly
```
VS Code nepieciešami Python un Jupyter paplašinājumi; izvēlieties atbilstošo Python kernel.

**Colab:** atveriet ar pogu augšā un izvēlieties *Runtime → Run all*. Ja Lecture 6 CSV lokāli nav atrodams, notebook pats nolasa to no GitHub.

Nākamā šūna instalē tikai trūkstošās pakotnes.

In [2]:
import importlib.util, subprocess, sys
packages = {"pandas":"pandas", "matplotlib":"matplotlib", "plotly":"plotly"}
missing = [pip for module,pip in packages.items()
           if importlib.util.find_spec(module) is None]
if missing:
    subprocess.check_call([sys.executable, "-m", "pip", "install", *missing])
print("Vide gatava.")

Vide gatava.


## 1. Importi un datu ielāde

Vispirms mēģinām atrast `apartment_ads.csv` klonētajā repozitorijā. Colab vai citā mapē izmantojam stabilu GitHub RAW adresi. Tādējādi analīzes kods pēc ielādes abās vidēs ir identisks.

In [3]:
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

DATA_URL = ("https://raw.githubusercontent.com/ValRCS/RTU_BDAA_Course_2026/"
            "main/notebooks/lecture_06_web_scraping/data/ss_flats_riga_centre_sell_20260916_190746.csv")
# DATA_URL = ("https://raw.githubusercontent.com/ValRCS/RTU_BDAA_Course_2026/"
#             "main/notebooks/lecture_06_web_scraping/apartment_ads.csv")
candidates = [
    Path("../lecture_06_web_scraping/data/ss_flats_riga_centre_sell_20260916_190746.csv"),
    Path("../lecture_06_web_scraping/apartment_ads.csv"),
    Path("notebooks/lecture_06_web_scraping/apartment_ads.csv"),
    Path("apartment_ads.csv"),
]
local = next((p for p in candidates if p.exists()), None)
source = local if local is not None else DATA_URL
raw_df = pd.read_csv(source)
print("Avots:", source)
print("Forma:", raw_df.shape)

Avots: ..\lecture_06_web_scraping\data\ss_flats_riga_centre_sell_20260916_190746.csv
Forma: (872, 9)


## 2. Pirmā apskate un kvalitātes pārbaude

Pirms tīrīšanas neko nemainām. `head()`, `sample()`, `info()` un `isna()` palīdz saprast, ko patiesībā ieguvām. Papildus pārbaudām sagaidāmo shēmu un dublētus URL — CSV veiksmīga atvēršana vēl negarantē korektus scraping datus.

In [ ]:
expected = {"url","description","Street","R.","m²","Floor","Series","Price, m2","Price"}
missing_cols = expected - set(raw_df.columns)
if missing_cols:
    raise ValueError("Trūkst kolonnu: " + ", ".join(sorted(missing_cols)))

display(raw_df.head())
display(raw_df.sample(min(5,len(raw_df)), random_state=42))
raw_df.info()
display(raw_df.isna().sum().to_frame("missing"))
print("Dublēti URL:", raw_df.duplicated(subset=["url"]).sum())
display(raw_df[["Price, m2","Price"]].head(10))

## 3. Kolonnu nosaukumi un datu nozīme

Analīzei izveidojam kopiju ar konsekventiem nosaukumiem. `rooms` un `area_m2` ir skaitliski lauki, `building_series` ir kategorija, bet `floor` un cenas pagaidām ir strukturēts teksts. Īpaši svarīgi: `Price` satur gan skaitli, gan mērvienību (`€/mon.`, `€/day`).

In [ ]:
df = raw_df.rename(columns={
    "Street":"street", "R.":"rooms", "m²":"area_m2", "Floor":"floor",
    "Series":"building_series", "Price, m2":"price_per_m2_raw",
    "Price":"price_raw"
}).copy()
display(df.head(3))

## 4. Teksts → skaitļi un mērvienības

Cena var būt `10 €`, `10.48€` vai `1,800 €/mon.`. Tāpēc ar `astype(float)` nepietiek. Funkcija normalizē simbolus un ar regulāro izteiksmi izvelk skaitli; `errors="coerce"` problemātisku vērtību pārvērš par `NaN`.

Mērvienību glabājam atsevišķi. `35 €/day` nav tieši salīdzināms ar `650 €/mon.` — skaitlis bez mērvienības var būt maldinošs.

In [ ]:
def euro_number(s):
    x = (s.astype("string").str.replace("\u00a0"," ",regex=False)
         .str.replace("€","",regex=False).str.replace(",","",regex=False)
         .str.replace(r"\s+","",regex=True)
         .str.extract(r"([-+]?\d+(?:\.\d+)?)", expand=False))
    return pd.to_numeric(x, errors="coerce")

def price_period(v):
    t = str(v).lower()
    if "/mon" in t or "/month" in t: return "month"
    if "/day" in t: return "day"
    if "/week" in t: return "week"
    if "/year" in t: return "year"
    return "unknown"

df["rooms"] = pd.to_numeric(df["rooms"], errors="coerce").astype("Int64")
df["area_m2"] = pd.to_numeric(df["area_m2"], errors="coerce")
df["price_per_m2"] = euro_number(df["price_per_m2_raw"])
df["price_eur"] = euro_number(df["price_raw"])
df["price_period"] = df["price_raw"].apply(price_period)

display(df[["price_raw","price_eur","price_period","price_per_m2"]].head(12))
display(df["price_period"].value_counts().to_frame("listings"))

## 5. `Floor` sadalīšana un feature engineering

`4/8` nozīmē “4. stāvs no 8”. No viena teksta lauka izveidojam `floor_number`, `building_floors`, `is_ground_floor` un `is_top_floor`. Šādu jaunu pazīmju veidošanu no esošiem datiem sauc par **feature engineering**.

In [ ]:
parts = df["floor"].astype("string").str.extract(
    r"(?P<floor_number>\d+)\s*/\s*(?P<building_floors>\d+)")
df["floor_number"] = pd.to_numeric(parts["floor_number"], errors="coerce").astype("Int64")
df["building_floors"] = pd.to_numeric(parts["building_floors"], errors="coerce").astype("Int64")
df["is_ground_floor"] = df["floor_number"].eq(1)
df["is_top_floor"] = df["floor_number"].eq(df["building_floors"])
df["building_series"] = df["building_series"].astype("string").str.strip().astype("category")

before = len(df)
df = df.drop_duplicates(subset=["url"], keep="first").copy()
df["calculated_price_per_m2"] = df["price_eur"] / df["area_m2"]
df["price_m2_difference"] = (df["price_per_m2"]-df["calculated_price_per_m2"]).abs()
print("Noņemti dublēti URL:", before-len(df))
display(df[["floor","floor_number","building_floors","is_top_floor"]].head(10))

## 6. Salīdzināma analīzes kopa

Galvenajai analīzei izmantojam tikai **mēneša īres** sludinājumus ar svarīgākajiem skaitliskajiem laukiem. Dienas cenu automātiska reizināšana ar 30 būtu biznesa pieņēmums, nevis neitrāla tīrīšana.

Tad demonstrējam tipisko Pandas plūsmu: **filtrēšana → kārtošana → aprakstošā statistika**.

In [ ]:
monthly_df = (df[df["price_period"]=="month"]
              .dropna(subset=["rooms","area_m2","price_eur","price_per_m2"])
              .copy())
print("Mēneša īres rindas:", len(monthly_df))

affordable = monthly_df[
    (monthly_df["rooms"]>=2) & (monthly_df["price_eur"]<=700)
].sort_values(["price_eur","area_m2"])
display(affordable[["street","rooms","area_m2","price_eur","price_per_m2"]].head(15))
display(monthly_df[["rooms","area_m2","price_eur","price_per_m2"]].describe().round(2))
print("Vidējā cena:", round(monthly_df["price_eur"].mean(),2))
print("Mediāna:", round(monthly_df["price_eur"].median(),2))

## 7. Pirmās vizualizācijas

**Histogramma** parāda nepārtraukta skaitliska mainīgā sadalījumu; **stabiņu diagramma** ir piemērota diskrētām kategorijām. Šeit skatām mēneša cenu sadalījumu un sludinājumu skaitu pēc istabu skaita.

In [ ]:
plt.figure(figsize=(9,5))
plt.hist(monthly_df["price_eur"], bins=30)
plt.xlabel("Mēneša īre (€)"); plt.ylabel("Sludinājumu skaits")
plt.title("Mēneša īres cenu sadalījums"); plt.tight_layout(); plt.show()

room_counts = monthly_df["rooms"].value_counts().sort_index()
plt.figure(figsize=(8,5))
plt.bar(room_counts.index.astype(str), room_counts.values)
plt.xlabel("Istabu skaits"); plt.ylabel("Sludinājumu skaits")
plt.title("Sludinājumi pēc istabu skaita"); plt.tight_layout(); plt.show()

## 8. `groupby()` un `agg()`

`groupby()` domāšanas modelis ir **split → apply → combine**: sadalām rindas grupās, katrai grupai aprēķinām statistiku un apvienojam rezultātus. Ar `agg()` vienlaikus iegūstam sludinājumu skaitu, mediānu, vidējo cenu, platību un `€/m²`.

In [ ]:
room_stats = (monthly_df.groupby("rooms")
    .agg(listings=("price_eur","count"),
         median_price=("price_eur","median"),
         mean_price=("price_eur","mean"),
         mean_area=("area_m2","mean"),
         mean_price_m2=("price_per_m2","mean"))
    .sort_index())
display(room_stats.round(2))

plt.figure(figsize=(8,5))
plt.bar(room_stats.index.astype(str), room_stats["median_price"])
plt.xlabel("Istabu skaits"); plt.ylabel("Mediānas īre (€)")
plt.title("Mediānas īres cena pēc istabu skaita"); plt.tight_layout(); plt.show()

## 9. Grupēšana pēc ēkas tipa

Kategorijām salīdzinām `€/m²`, bet vienmēr skatām arī grupas lielumu: viena sludinājuma “mediāna” ir daudz mazāk uzticama par simtiem novērojumu. Gariem nosaukumiem izmantojam horizontālu stabiņu diagrammu.

In [ ]:
series_stats = (monthly_df.groupby("building_series", observed=True)
    .agg(listings=("price_per_m2","count"),
         median_price_m2=("price_per_m2","median"),
         mean_price_m2=("price_per_m2","mean"))
    .sort_values("median_price_m2"))
display(series_stats.round(2))

plt.figure(figsize=(9,6))
plt.barh(series_stats.index.astype(str), series_stats["median_price_m2"])
plt.xlabel("Mediānas cena par m² (€)"); plt.ylabel("Ēkas sērija / tips")
plt.title("Cena par m² pēc ēkas tipa"); plt.tight_layout(); plt.show()

## 10. Scatter plot, korelācija un boxplot

Scatter plot palīdz ieraudzīt saistību starp platību un cenu. `corr()` apkopo lineāras saistības, bet **korelācija nepierāda cēloņsakarību**. Boxplot savukārt rāda mediānu, kvartiles, izkliedi un potenciālus outlier, tādēļ grupu salīdzinājumam saglabā vairāk informācijas nekā viena vidējā vērtība.

In [ ]:
plt.figure(figsize=(9,6))
plt.scatter(monthly_df["area_m2"], monthly_df["price_eur"], alpha=0.6)
plt.xlabel("Platība (m²)"); plt.ylabel("Mēneša īre (€)")
plt.title("Platība un mēneša īres cena"); plt.tight_layout(); plt.show()

corr_cols = ["rooms","area_m2","price_eur","price_per_m2","floor_number","building_floors"]
display(monthly_df[corr_cols].astype(float).corr().round(3))

plt.figure(figsize=(9,6))
monthly_df.boxplot(column="price_per_m2", by="rooms", grid=False)
plt.xlabel("Istabu skaits"); plt.ylabel("Cena par m² (€)")
plt.title("Cena par m² pēc istabu skaita"); plt.suptitle("")
plt.tight_layout(); plt.show()

## 11. Praktisks jautājums un neparasti novērojumi

Meklējam 2 istabu dzīvokļus ar vismaz 40 m² un īri līdz 700 €, kārtojot pēc `€/m²`. Pēc tam apskatām dārgākos ierakstus.

**Outlier nav automātiski kļūda.** Tas var būt luksusa objekts, atšķirīgs piedāvājuma tips vai scraping problēma; to pārbauda, atverot avota URL.

In [ ]:
choice = (monthly_df[
    (monthly_df["rooms"]==2) &
    (monthly_df["area_m2"]>=40) &
    (monthly_df["price_eur"]<=700)
].sort_values(["price_per_m2","price_eur"]))
cols = ["street","area_m2","floor","building_series","price_eur","price_per_m2","url"]
display(choice[cols].head(20))

print("Augstākās cenas par m²:")
display(monthly_df.nlargest(10,"price_per_m2")[
    ["street","rooms","area_m2","price_eur","price_per_m2","url"]])

## 12. Reproducējama tīrīšanas funkcija un eksports

Lekcijā tīrīšanu veicām pa soļiem, lai redzētu katras darbības nozīmi. Automatizācijā šos soļus apvieno funkcijā, kuru var atkārtoti palaist nākamajam scraping rezultātam. Attīrīto kopu saglabājam atsevišķi no neapstrādātajiem datiem.

In [ ]:
def clean_apartments(source_df):
    x = source_df.rename(columns={
        "Street":"street","R.":"rooms","m²":"area_m2","Floor":"floor",
        "Series":"building_series","Price, m2":"price_per_m2_raw","Price":"price_raw"
    }).copy()
    x["rooms"] = pd.to_numeric(x["rooms"],errors="coerce").astype("Int64")
    x["area_m2"] = pd.to_numeric(x["area_m2"],errors="coerce")
    x["price_per_m2"] = euro_number(x["price_per_m2_raw"])
    x["price_eur"] = euro_number(x["price_raw"])
    x["price_period"] = x["price_raw"].apply(price_period)
    p = x["floor"].astype("string").str.extract(
        r"(?P<floor_number>\d+)\s*/\s*(?P<building_floors>\d+)")
    x["floor_number"] = pd.to_numeric(p["floor_number"],errors="coerce").astype("Int64")
    x["building_floors"] = pd.to_numeric(p["building_floors"],errors="coerce").astype("Int64")
    x["is_ground_floor"] = x["floor_number"].eq(1)
    x["is_top_floor"] = x["floor_number"].eq(x["building_floors"])
    x["building_series"] = x["building_series"].astype("string").str.strip().astype("category")
    x = x.drop_duplicates(subset=["url"], keep="first").copy()
    x["calculated_price_per_m2"] = x["price_eur"]/x["area_m2"]
    return x

clean_df = clean_apartments(raw_df)
OUTPUT = Path("apartments_clean.csv")
clean_df.to_csv(OUTPUT, index=False, encoding="utf-8")
print("Saglabāts:", OUTPUT.resolve(), "| rindas:", len(clean_df))

## 13. Praktiskie uzdevumi

1. Atrodiet platības mediānu un piecus lētākos sludinājumus.
2. Saskaitiet sludinājumus pēc istabu skaita.
3. Aprēķiniet mediānas īri pēc istabu skaita.
4. Aprēķiniet mediānas `€/m²` pēc ēkas tipa.
5. Uzzīmējiet platības histogrammu.
6. Uzzīmējiet mediānas īri pēc istabu skaita.
7. Izveidojiet scatter plot: platība pret cenu.
8. Atrodiet 10 augstākās `€/m²` vērtības.
9. Definējiet savus atlases kritērijus un atrodiet 5 kandidātus.

Šūnā zemāk ir komentēti sākumpunkti, tādēļ **Run All** paliek drošs.

In [ ]:
# monthly_df["area_m2"].median()
# monthly_df.nsmallest(5, "price_eur")
# monthly_df["rooms"].value_counts().sort_index()

# my_choice = monthly_df[
#     (monthly_df["rooms"] >= 2)
#     & (monthly_df["area_m2"] >= 45)
#     & (monthly_df["price_eur"] <= 750)
# ]
# display(my_choice.sort_values("price_per_m2").head(5))

## 14. Bonuss — Plotly

Matplotlib paliek lekcijas galvenā bibliotēka. Plotly īsi demonstrē interaktīvu scatter plot ar hover informāciju, zoom un krāsu pēc istabu skaita.

In [ ]:
import plotly.express as px
plot_df = monthly_df.dropna(subset=["area_m2","price_eur","rooms"]).copy()
plot_df["rooms_label"] = plot_df["rooms"].astype("string")
fig = px.scatter(
    plot_df, x="area_m2", y="price_eur", color="rooms_label",
    hover_data=["street","building_series","price_per_m2"],
    labels={"area_m2":"Platība (m²)","price_eur":"Mēneša īre (€)",
            "rooms_label":"Istabas"},
    title="Rīgas centra dzīvokļi: platība un mēneša īres cena")
fig.show()

## Kopsavilkums

**SS.com → scraping → CSV → Pandas → tīrīšana → analīze → vizualizācija → secinājumi → tīrs CSV**

Svarīgākais: analīze sākas ar datu apskati; skaitlis bez mērvienības var būt kļūdaini interpretējams; `groupby()` + `agg()` ir centrāls biznesa analīzes paņēmiens; dažādi grafiki atbild uz dažādiem jautājumiem; outlier nav automātiski kļūda; tīrīšanas loģiku vērts padarīt atkārtojamu.